__Interactive Brokers (IBKR) API Tools__

> __Interactive Brokers__ is an established and respected broker that you can use for trading various asset like stocks, options, and futures. They provide a trading platform known as Trader Workstation (__TWS__) that allows you to manually place trades, both on the web and through a desktop app. Additionally, IBKR provides an API that allows you to interact programmatically with your account, allowing trades to be placed and historical and real-time data to be accessed. What follows will be some tools I've developed to more easily gather data and help others get started using their low-level API. IBKR offers many education resources on how to use their API and should be your primary reference; this notebook is simply to help you get started and provide some additional utilities you may find useful. Further, it should be noted that you need an account with IBKR to gain access and any necessary data subscriptions, particularly for real-time streaming. Assuming you have installed the API per the directions on their site and have an account, the following code should work for you. As a final note, nothing in what follows should be taken as financial or investment advice, and is for educational purposes only. Note that no personal information or account details are required to connect to the API, and none will be present in this notebook or code. When connecting, you must have a running instance of TWS or IB Gateway. Your code can then connect to that instance locally.

I want to bring to attention a Python package called _ib_insync_. This is a high-level wrapper around _ibapi_ that simplifies usage of the API, and in most cases is the preferable approach. However, the low-level API allows more control and flexibility when handling your data. While  _ib_insync_ might be the best solution for your use case, I think it's useful to know how things work under the hood. Plus, there are other utilities I will provide that are independent of the API itself which you might find useful when generating your own time series data to experiment with.

Some of the functions I give can be used to help clean your data and ensure it behaves as expected. Market data is notoriously difficult to deal with for a variety of reasons, such as missing dates, holidays, early closures, poor liquididty leading to missing data, etc. So some of these tools can help you ensure the dataset you're building is consistent and appropriate for your use case, allowing you to avoid any 'gotchas'.

The API can be imported in Python as follows:

In [3]:
import ibapi

SyntaxError: invalid syntax (1381723849.py, line 1)

All helper functions and class definitions are stored in _ibkr_helpers.py_ and imported in this walkthrough. Further, we import some additional modules to help with the demonstration of some of the tools. Note there is a Python script called _ibkr_walthrough_script.py_ that contains the code in this notebook.

In [2]:
import time
from datetime import datetime, timedelta
import mplfinance as mpf
import ibkr_helpers

ModuleNotFoundError: No module named 'mplfinance'

_ibapi_ works by receiving data asynchronously via callback methods in a class definition. When you request the data, it will be delivered only after the client is running. However, running the client is a blocking call, disallowing further script execution. Therefore, we must run the client in a separate thread. I provide two class definitions that are templates you can modify to your needs: one for obtaining historical data __(IBAppHistoricalBars)__ and another for real-time streaming data __(IBAppBidAskStreamer)__. A few of the utility functions call on these and we need not access them directly.

The following code will obtain the last 30 days of daily OHLCV (Open, High, Low, Close, and Volume) bar data for the tickers indicated and print the most recent 5 for each. This function currently only allows daily bars (_'1 day'_) or minute bars (_'1 min_'), but you could modify it easily to accept other timeframes.

In [ ]:
symbols_daily = ['TSLA', 'GOOG', 'MSFT']
daily_results, failed_tickers_daily = ibkr_helpers.get_last_n_days_ohlcv(30,
                                                                         symbols_daily,
                                                                         '1 day')
 
for symb_daily in symbols_daily:
    if symb_daily not in failed_tickers_daily:
        print(f'Daily data for symbol {symb_daily}')
        print(daily_results[symb_daily].tail(5))
        print('-'*50)

Here is a sample output from the code above:

Now, let's grab some minute bar data for two other stocks, again returned as pandas DataFrames. A sample output is in the block that follows.

In [4]:
symbols_min = ['AA', 'KO']
minute_results, failed_tickers_min = ibkr_helpers.get_last_n_days_ohlcv(1,
                                                                        symbols_min,
                                                                        '1 min')

for symb_min in symbols_min:
    if symb_min not in failed_tickers_min:
        print(f'Minute data for symbol {symb_min}')
        print(minute_results[symb_min].tail(5))
        print('-'*50)

NameError: name 'ibkr_helpers' is not defined

The API allows you access to other information for a given asset, such as the exchange it is traded on and what industry it belongs to. This information could be useful when organinzing your data and ensuring no one particular feature is overrepresented. In the __IBAppHistoricalBars__ class I demonstrate how we can obtain such information. Again, a sample output is in the block that follows.

In [ ]:
symbol = 'TSLA'

# Initialize IB API
app = ibkr_helpers.IBAppHistoricalBars()
app.start() # uses default paper args.

# Wait until the connection is established
print("Waiting for connection...")

app.connected_event.wait(timeout = 10)  # Wait for connection (max 10 seconds).
    
if not app.connected_event.is_set():
    print("Failed to establish connection.")
else:  
    contract = ibkr_helpers.make_contract(symbol)
    con_req_id = app.get_next_contract_req_id()
    app.contract_reqId_to_symbol[con_req_id] = symbol
    app.reqContractDetails(con_req_id, contract) # This makes ticker_to_exchange and ticker_to_industry dicts in the app internally.
    time.sleep(5) # Give it a few seconds to retrieve the information.
    print(f'{symbol} exchange = ', app.ticker_to_exchange[symbol])
    print(f'{symbol} industry = ', app.ticker_to_industry[symbol])
    app.disconnect()

Now let's utilize the streaming class __IBAppBidAskStreamer__ via the helper function _get_streamer_bid_ask_app_ to get real time 5-second updated bars for the asset. This must be run during active market hours. In this case I chose to display the bid and ask prices, but you could easily modify it to show standard candlesticks as well.

In [ ]:
# How long do you want to stream?
observe_for_sec = 60

symbol = 'TSLA'
bid_req_id = 1
ask_req_id = bid_req_id + 1

app = ibkr_helpers.get_streamer_bid_ask_app(symbol,
                                            bid_req_id,
                                            ask_req_id)

if app:
    st_time = time.time()
    while time.time() - st_time < observe_for_sec:
        last_bid = round(app.bid_df['close_bid'].iloc[-1], 2)
        last_ask = round(app.ask_df['close_ask'].iloc[-1], 2)
        spread = last_ask - last_bid
        spread_perc = round(100*spread/last_bid, 2)
        print(f"Last bid was ${last_bid:,}")
        print(f"Last ask was ${last_ask:,}")
        print(f"Dollar Spread is ${spread:,}")
        print(f"Ask is {spread_perc}% above the bid.")
        time.sleep(5) # 5 second bars, so wait for a new bar to be received.
        
    app.disconnect_app_and_stream()

Another helper function I thought I'd shareis the calculation of Heikin-Ashi candles from regular candles. We can obtain regular candle bars from _get_last_n_days_ohlcv_ and use those to obtain the Heikin-Ashi candles, which are essentially a more smoothed out representation of the price action. Let's use the Tesla candles from earlier and plot the regular candles above the calulated Heikin-Ashi candles. Once again, an example output will follow the code block.

In [ ]:
tsla_daily_bars = daily_results['TSLA']
tsla_daily_bars_ha = ibkr_helpers.calculate_heikin_ashi(tsla_daily_bars)

# Rename columns to match mplfinance expectations.
ohlc = tsla_daily_bars.rename(columns={
    'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close'
})

ha = tsla_daily_bars_ha.rename(columns={
    'open_h': 'Open', 'high_h': 'High', 'low_h': 'Low', 'close_h': 'Close'
})

# Create the Heikin-Ashi addplot on panel 1
ha_plot = mpf.make_addplot(
    ha[['Open', 'High', 'Low', 'Close']],
    type='candle',
    panel=1
)

# Plot both panels in one call
mpf.plot(
    ohlc,
    type = 'candle',
    addplot = ha_plot,
    panel_ratios = (3, 2),
    style = 'yahoo',
    title = 'TSLA: OHLC (Top) & Heikin-Ashi (Bottom)',
    show_nontrading = False,
    volume = False
)

<div style="text-align: center;">
    <img src="reg_and_ha_tsla.png" alt="TSLA Candles">
</div>

Sometimes you will need to know when a symbol was first made publicly available for trading. More precisely, you want to know the earliest date you can obtain data for the symbol. Since each symbol has a different start date, I have provided a helper to extract this date given a ticker symbol. Note that since the markets have been around for many decades, the start date returned will not necessarily be the true IPO date. For example, __KO__ (The Coca-Cola Company) first traded in September of 1919, but digital record only go back to January of 1962.

In [6]:
ko_first_date = ibkr_helpers.get_symbols_first_date('KO')
print(f'Coca-Cola first started trading on {ko_first_date} in YYYYMMDD format.')

NameError: name 'ibkr_helpers' is not defined

The following function will return the last _n_ trading days looking back from today. This can be useful to use as a sanity check to ensure there are no missing dates due to temporary sever downtime or disconnection issues. Basically, it's another tool t oensure you are working with the best dataset you can get without any unexpectd anomalies. It utilizes a function that calls a recursive function while making sure no holidays or full closure days are counted. Today is 20250827 after market hours and calling this function with _n_ = 60 yields the following results.

In [ ]:
last_trading_days = ibkr_helpers.get_last_n_trading_days_from_now(60)
print('The last trading days were: ', last_trading_days)

To supplement this functionality, we can extract all days that were full market closure days (exclusing weekends since the market is never open on weekends). To do this, we must specify how many days in the past we want to search and how many days forward into the future. the output, ran today (20250827) is as follows.

In [ ]:
days_back = 90
days_forward = 7
closures = ibkr_helpers.get_full_closures(days_back = days_back,
                                          days_forward = days_forward)

print('The full closure days are: ', closures)

Note these days are excluded from the variable _last_trading_days_ above and includes '20250901', which is Labor Day.

Depending on your goal, it might be useful knowing when the market closed, or will close, early. The following example behaves similar to _get_full_closures_ and returns a dictionary mapping days (in 'YYYYMMDD' format) to early close time (in 'HHMM' format). E.g. '1300' means 1:00 PM ET.

In [ ]:
days_back = 90
days_forward = 7
early_closures = ibkr_helpers.get_early_market_closures(days_back = days_back,
                                                        days_forward = days_forward)

print('The early closure days (YYYYMMDD format) and closure times are: ', early_closures)

Here we can see that on July 3, 2025 the market closes early at 1:00 PM ET and no early closures are planned at least 7 days into the future (from today).

As a final helper function, it can be useful to know precisely how many trading days have occurred given a certain start date (in 'YYYYMMDD' format). The function will not count that start date itself. For example, if today is 202508827 and the market has closed, and I ask for the number of trading days that have occurred starting 10 calendar days ago to now, I get 8. The variable _past_date_str_ is set to '20250817' and counts the trading days up to and including today.

In [ ]:
past_date = datetime.today() - timedelta(days = 10)
past_date_str = past_date.strftime('%Y%m%d')

# We use closures and early_closures above, which will work because days_back = 90 > 70.
completed_trading_days = ibkr_helpers.count_completed_trading_days(past_date_str,
                                                                   closures,
                                                                   early_closures)

print(f'The number of completed trading days since, but not including, {past_date_str} is {completed_trading_days}.')


Thank you for taking the time to read through the notebook!